In [1]:
!pip install wordninja

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.6/541.6 kB 10.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wordninja: filename=wordninja-2.0.0-py3-none-any.whl size=541553 sha256=a5847bb002680c3957b25b1e0986ef2d454325523e0e44b64c217cc7f74bfcc1
  Stored in directory: /Users/viv/Library/Caches/pip/wheels/8c/28/16/1ee6c4de039d112b6d448c7aa61f0f62c27cfcd0d2b08762fb
Successfully built wordninja


In [3]:
import json
import re
import unicodedata
from pathlib import Path
import wordninja

# Config: Convert all newlines (\r, \n) into a single space for dense vector embedding models.
# Set to False if you prefer keeping clean \n\n paragraph breaks.
FLATTEN_NEWLINES = True

# Match 4+ individual characters separated by whitespace (e.g. "l u n g i s s u e s")
MONOSPACE_RUN_PATTERN = re.compile(r"(?:\b[a-zA-Z0-9]\s+){3,}[a-zA-Z0-9]\b")

# Noise patterns
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
PAGE_NUM_PATTERN = re.compile(r"(?:(?<=\n)|^)\s*\d+\s*(?=\n|$)")

# Cyrillic OCR corruption transliteration map
CYRILLIC_HOMOGLYPHS = str.maketrans({
    "У": "Y", "у": "y",
    "е": "e",
    "Т": "T", "т": "t",
    "а": "a",
    "к": "k",
    "р": "p",
    "о": "o",
    "с": "c",
})

# Seed Ayurvedic/Yogic vocabulary into wordninja's unigram cost table
AYURVEDA_VOCAB = [
    "kundalini", "mooladhara", "swadhisthana", "manipura", "anahata",
    "vishuddhi", "sahasrara", "pranayama", "sushumna", "tapasya", "aushadhi",
    "shaktipat", "shatkarma", "tattwa", "kriya", "sadhaka", "sadhana",
    "brahma", "vajrini", "chitrini", "samadhi", "moksha", "shiva", "shakti",
    "dosha", "vata", "pitta", "kapha", "dhatus", "ojas", "agni", "ama"
]

lm = wordninja.DEFAULT_LANGUAGE_MODEL
for word in AYURVEDA_VOCAB:
    w = word.lower()
    lm._wordcost[w] = 0.0  # Lowest cost -> highest probability during Viterbi split
    if len(w) > lm._maxword:
        lm._maxword = len(w)

print(f"Config initialized. Added {len(AYURVEDA_VOCAB)} domain terms to wordninja language model.")

Config initialized. Added 32 domain terms to wordninja language model.


In [4]:
def heal_monospace_spaced_text(text: str) -> str:
    """Detects spaced-out monospace sequences, strips gaps, and re-segments words."""
    def replacer(match: re.Match) -> str:
        raw_match = match.group(0)
        collapsed = re.sub(r"\s+", "", raw_match)
        tokens = wordninja.split(collapsed)
        return " ".join(tokens)

    return MONOSPACE_RUN_PATTERN.sub(replacer, text)

def clean_chunk_content(text: str) -> str:
    # 1. Unicode canonical normalization
    text = unicodedata.normalize("NFKC", text)

    # 2. Fix OCR Cyrillic artifacts
    text = text.translate(CYRILLIC_HOMOGLYPHS)

    # 3. Strip URLs and raw page number markers
    text = URL_PATTERN.sub("", text)
    text = PAGE_NUM_PATTERN.sub("", text)

    # 4. Resolve monospaced/spaced-out text
    text = heal_monospace_spaced_text(text)

    # 5. Handle newline formatting
    if FLATTEN_NEWLINES:
        text = re.sub(r"[\r\n]+", " ", text)
    else:
        # Re-attach lines wrapped mid-sentence
        text = re.sub(r"(?<=[^\n\.?!:])\n(?=[a-z])", " ", text)
        # Collapse multiple blank lines down to standard double-newline
        text = re.sub(r"\n{3,}", "\n\n", text)

    # 6. Normalize horizontal whitespace
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [5]:
SOURCE_DIR = Path("/Users/viv/Downloads/Ayurveda")
INPUT_PATH = SOURCE_DIR / "result.json"
OUTPUT_PATH = SOURCE_DIR / "cleanedresult.json"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Missing file at: {INPUT_PATH}")

print(f"Loading {INPUT_PATH}...")
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} chunks. Processing content...")

# Step 1: Clean each chunk's content
for item in data:
    item["content"] = clean_chunk_content(item.get("content", ""))

# Step 2: Stitch broken words cut across chunk boundaries
stitched_count = 0
for i in range(len(data) - 1):
    curr_text = data[i]["content"]
    next_text = data[i + 1]["content"]

    m_curr = re.search(r"(\b[A-Za-z]+)$", curr_text)
    m_next = re.match(r"^([a-z]+)\b", next_text)

    if m_curr and m_next and not curr_text.endswith((".", "!", "?", '"', ":", ";")):
        broken_word_end = m_next.group(1)
        data[i]["content"] = curr_text + broken_word_end
        data[i + 1]["content"] = next_text[len(broken_word_end):].lstrip()
        stitched_count += 1

print(f"Stitched {stitched_count} words severed across chunk borders.")

# Step 3: Write out to cleanedresult.json
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"Cleaned dataset saved to: {OUTPUT_PATH}")

Loading /Users/viv/Downloads/Ayurveda/result.json...
Loaded 27447 chunks. Processing content...
Stitched 6953 words severed across chunk borders.
Cleaned dataset saved to: /Users/viv/Downloads/Ayurveda/cleanedresult.json


In [6]:
# Inspect the first few processed chunks
with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
    inspected_data = json.load(f)

for chunk in inspected_data[:3]:
    print(f"ID: {chunk['id']}")
    print(f"Sample Content: {chunk['content'][:120]}...")
    print(f"Contains '\\n': {'\\n' in chunk['content']}")
    print("-" * 60)

ID: Kundalini_Tantra_(Swami_Satyananda_Saraswati)_(Z-Library)-chunk-0
Sample Content: "Kundalini Tantra" Swami Satyananda Saraswati CONTENTS Introduction to Kundalini Tantra Section I - KUNDALINI 1. Ye Man,...
Contains '\n': False
------------------------------------------------------------
ID: Kundalini_Tantra_(Swami_Satyananda_Saraswati)_(Z-Library)-chunk-1
Sample Content: 5. Methods of Awakening 6. Preparing for the Awakening 7. Diet for Kundalini Awakening 8. Risks and Precautions 9. Kunda...
Contains '\n': False
------------------------------------------------------------
ID: Kundalini_Tantra_(Swami_Satyananda_Saraswati)_(Z-Library)-chunk-2
Sample Content: 12. The Experiences of Awakening 13. The Path of Kriya Yoga 14. Vama Marga and Kundalini Awakening Section 2 - THE CHAKR...
Contains '\n': False
------------------------------------------------------------
